# WOVO — Remote Ollama Server via Cloudflare Tunnel

Runs Ollama on Colab's free T4 GPU and exposes it via `cloudflared` tunnel.

**Advantages over ngrok:**
- No auth token or signup needed (quick tunnel)
- No rate limits on the tunnel itself
- More stable connections
- Option to use your own domain (e.g., `ollama.vishalk.com`)

**Before running:** Go to **Runtime → Change runtime type → T4 GPU**

**Two tunnel modes:**
- **Cell 3a** — Quick tunnel (zero config, random `*.trycloudflare.com` URL)
- **Cell 3b** — Named tunnel on your domain (e.g., `ollama.vishalk.com`, requires one-time setup)

In [1]:
#@title Cell 1 — Check GPU & Install Ollama + cloudflared
!nvidia-smi

!apt-get update -qq && apt-get install -y -qq zstd pciutils > /dev/null

print("\n" + "="*50)
print("Installing dependencies + Ollama...")
print("="*50)
!apt-get update -qq && apt-get install -y -qq zstd pciutils > /dev/null
!curl -fsSL https://ollama.ai/install.sh | sh
!ollama --version

print("\n" + "="*50)
print("Installing cloudflared...")
print("="*50)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!cloudflared --version

/bin/bash: line 1: nvidia-smi: command not found
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

Installing dependencies + Ollama...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
##                                                                         3.0%^C

Installing cloudflared...
Selecting previously unselected package cloudflared.
(Reading database ... 118237 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...
cloudflared version 2026.3.0 (built 2026-03-09-14

In [2]:
#@title Cell 2 — Start Ollama & Pull Models
import subprocess
import time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

models = [
    ("qwen3:4b",  "Text generation — survey analysis, clustering, forecasts, voice analytics"),
    ("gemma3:1b", "Lightweight — payslip anomaly interpretation"),
    ("bge-m3",    "Embeddings — case clustering (768-dim vectors)"),
]

for model_name, purpose in models:
    print(f"\n{'='*50}")
    print(f"Pulling {model_name} — {purpose}")
    print(f"{'='*50}")
    !ollama pull {model_name}

print("\n\u2705 All models ready!")
!ollama list


Pulling qwen3:4b — Text generation — survey analysis, clustering, forecasts, voice analytics
^C

Pulling gemma3:1b — Lightweight — payslip anomaly interpretation
Error: could not connect to ollama server, run 'ollama serve' to start it

Pulling bge-m3 — Embeddings — case clustering (768-dim vectors)
Error: could not connect to ollama server, run 'ollama serve' to start it

✅ All models ready!
Error: could not connect to ollama server, run 'ollama serve' to start it


In [ ]:
#@title Cell 3a — Quick Tunnel (zero config, random URL)
#@markdown Run this OR Cell 3b, not both.
#@markdown Gives you a random `*.trycloudflare.com` URL — changes each session.

import subprocess
import re
import time

# Start cloudflared quick tunnel in background
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for the tunnel URL to appear in output
url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.1)
        continue
    match = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
    if match:
        url = match.group(1)
        break

if url:
    print(f"\n{'='*60}")
    print(f"  Ollama is live! Set this in your .env.local:")
    print(f"")
    print(f"  OLLAMA_BASE_URL={url}/api")
    print(f"")
    print(f"  Quick test from your terminal:")
    print(f"  curl {url}/api/tags")
    print(f"{'='*60}")
else:
    print("ERROR: Tunnel did not start within 30s. Check output above.")

In [ ]:
#@title Cell 3b — Named Tunnel on vishalk.com (persistent subdomain)
#@markdown Run this OR Cell 3a, not both.
#@markdown
#@markdown **One-time setup (on your local machine, not Colab):**
#@markdown 1. `cloudflared tunnel login` (authenticates with Cloudflare)
#@markdown 2. `cloudflared tunnel create ollama` (creates tunnel, note the tunnel ID)
#@markdown 3. `cloudflared tunnel route dns ollama ollama.vishalk.com` (creates DNS record)
#@markdown 4. Copy `~/.cloudflared/<tunnel-id>.json` credentials file
#@markdown 5. Paste the JSON content below

import json
import subprocess
import os

# ============================================================
# PASTE YOUR TUNNEL CREDENTIALS JSON HERE
# (from ~/.cloudflared/<tunnel-id>.json on your local machine)
TUNNEL_CREDENTIALS = """
{
  "AccountTag": "",
  "TunnelSecret": "",
  "TunnelID": ""
}
"""

TUNNEL_HOSTNAME = "ollama.vishalk.com"  # Your subdomain
# ============================================================

creds = json.loads(TUNNEL_CREDENTIALS)
if not creds.get("TunnelID"):
    raise ValueError("Please fill in TUNNEL_CREDENTIALS above. See the setup instructions.")

tunnel_id = creds["TunnelID"]

# Write credentials file
os.makedirs(os.path.expanduser("~/.cloudflared"), exist_ok=True)
creds_path = os.path.expanduser(f"~/.cloudflared/{tunnel_id}.json")
with open(creds_path, "w") as f:
    json.dump(creds, f)

# Write config file
config = f"""tunnel: {tunnel_id}
credentials-file: {creds_path}

ingress:
  - hostname: {TUNNEL_HOSTNAME}
    service: http://localhost:11434
  - service: http_status:404
"""
config_path = os.path.expanduser("~/.cloudflared/config.yml")
with open(config_path, "w") as f:
    f.write(config)

print(f"Config written to {config_path}")
print(f"Starting tunnel to {TUNNEL_HOSTNAME}...\n")

# Start named tunnel
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "run", tunnel_id],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

import time
time.sleep(5)

print(f"{'='*60}")
print(f"  Ollama is live! Set this in your .env.local:")
print(f"")
print(f"  OLLAMA_BASE_URL=https://{TUNNEL_HOSTNAME}/api")
print(f"")
print(f"  Quick test from your terminal:")
print(f"  curl https://{TUNNEL_HOSTNAME}/api/tags")
print(f"{'='*60}")
print(f"\n  This URL is stable across sessions (same subdomain every time).")

In [ ]:
#@title Cell 4 — Keep Alive
import time
import requests

print("Keeping Colab session alive. Leave this cell running.")
print("Press the stop button (or Runtime \u2192 Interrupt) to shut down.\n")

tick = 0
while True:
    tick += 1
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        model_count = len(r.json().get("models", []))
        status = f"OK ({model_count} models loaded)"
    except Exception as e:
        status = f"ERROR: {e}"

    print(f"[{tick:>4}m] Ollama health: {status}", end="\r")
    time.sleep(60)